In [46]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import requests
from pathlib import Path
import os
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import database

In [34]:
API_KEY = os.environ['KEY']

In [28]:
null_dates = database.db_requester_fetchall(sql_script="get_null_dates.sql")

In [31]:
null_dates

[('2026-09',)]

In [140]:
null_dates[0][0]

'2025-10'

In [ ]:
# for testing (2025-10 is not available so FRED returns ".")
null_dates = [('2025-10',), ('2026-07',), ('2026-08',), ('2026-09',)]
null_dates

[('2025-10',), ('2026-07',), ('2026-08',), ('2026-09',)]

In [ ]:
for date in null_dates:
    print(date[0])
    print(datetime.strptime(date[0], "%Y-%m").strftime("%Y-%m-%d"))

2026-08
2026-08-01
2026-09
2026-09-01


# Get historical, unrevised values

In [ ]:
def actual_hist_unrates(null_dates):

    url = "https://api.stlouisfed.org/fred/series/observations"
    hist_unrates = []

    with requests.Session() as session:

        # Implement null_dates requester here
        
        for date in null_dates:
            params = {
                "api_key": API_KEY,
                "observation_start": datetime.strptime(date[0], "%Y-%m").strftime("%Y-%m-%d"),
                "observation_end": datetime.strptime(date[0], "%Y-%m").strftime("%Y-%m-%d"),
                "realtime_start": "1776-07-04",
                "realtime_end": "9999-12-31",
                "output_type": "4", # retrieves the initial release value and if no revisions were done, then the only available value
                "file_type": "json",
                "series_id": "UNRATE"
            }

            r = session.get(url=url, params=params, timeout=30)

            # print(r.url)
            # print(r.status_code)
            # print(r.text)

            r.raise_for_status()

            data = r.json()
            
                
            forecast_target_t1 = date[0]
            if len(data['observations']) != 0:
                def unrate_availability(unrate):
                    try:
                        return float(unrate)
                    except(ValueError, TypeError):
                        return None
                release_date_historical = data['observations'][0]['realtime_start']
                actual_unrate_historical = unrate_availability(data['observations'][0]['value'])
            else: release_date_historical, actual_unrate_historical = None, None

            hist_unrates.append({"forecast_target_t1": forecast_target_t1,
                                    "release_date_historical": release_date_historical,
                                    "actual_unrate_historical": actual_unrate_historical})

            #    print("Publication date: ", release_date_historical,
            #          "\t UNRATE for: ", forecast_target_t1,
            #          "\t UNRATE:", actual_unrate_historical)
    return hist_unrates


In [139]:
actual_hist_unrates(null_dates)

[{'forecast_target_t1': '2025-10',
  'release_date_historical': '2025-12-16',
  'actual_unrate_historical': None},
 {'forecast_target_t1': '2026-07',
  'release_date_historical': '2026-08-07',
  'actual_unrate_historical': 4.1},
 {'forecast_target_t1': '2026-08',
  'release_date_historical': '2026-09-04',
  'actual_unrate_historical': 4.1},
 {'forecast_target_t1': '2026-09',
  'release_date_historical': None,
  'actual_unrate_historical': None}]